# Extract and Rank Misrecognized Words from ASR Evaluations

In [4]:
!pip install -U -q jiwer pandas tqdm

## Load the Evaluation Results (TSV Files from the ASR Exercise)

In [2]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../mmai_07")

tsv_paths = sorted(RESULTS_DIR.glob("*without_context.tsv"))
print(tsv_paths)

results = []
for tsv_path in tsv_paths:
    df = pd.read_csv(tsv_path, sep="\t")
    df["source"] = tsv_path.stem
    results.append(df)

results = pd.concat(results, ignore_index=True)
results.head()

[PosixPath('../mmai_07/granite_finetuned_commonvoice_without_context.tsv'), PosixPath('../mmai_07/granite_finetuned_librispeech_without_context.tsv'), PosixPath('../mmai_07/granite_finetuned_voxpopuli_without_context.tsv')]


,audio_path,label,prediction,wer,cer,source
0,/training-1/commonvoice_all/cv-corpus-19/cv-co...,cristiano ronaldo has won twenty four trophies...,it is carried and the brand has been wanted tw...,1.300000,0.733333,granite_finetuned_commonvoice_without_context
1,/training-1/commonvoice_all/cv-corpus-19/cv-co...,i guess you must think im kinda batty,quem está mais velho há um problema,1.000000,0.783784,granite_finetuned_commonvoice_without_context
2,/training-1/commonvoice_all/cv-corpus-19/cv-co...,enoxaparin belongs to the class of drugs known...,hello good morning the oxpeacian belongs to th...,0.692308,0.531646,granite_finetuned_commonvoice_without_context
3,/training-1/commonvoice_all/cv-corpus-19/cv-co...,he was later killed by john wesley hardin duri...,he was later killed by john wesley harding dur...,0.333333,0.114286,granite_finetuned_commonvoice_without_context
4,/training-1/commonvoice_all/cv-corpus-19/cv-co...,the immersion in hot water also kills the silk...,the immersion in hot water also kills the sick...,0.300000,0.090909,granite_finetuned_commonvoice_without_context


## Text Normalization and Word-Level Alignment

## Named Entity Recognition on the Labels

In [ ]:
!pip install -U -q spacy
!python -m spacy download en_core_web_trf

In [ ]:
import spacy
import re
import string

nlp = spacy.load("en_core_web_trf")   # alternatives: de_dep_news_trf, fr_dep_news_trf, es_dep_news_trf

def normalize(text):
    text = str(text).translate(str.maketrans("", "", string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

def extract_entities(text):
    """Normalized named-entity strings from the label."""
    doc = nlp(str(text))
    return [normalize(ent.text) for ent in doc.ents if normalize(ent.text)]

print(extract_entities(results.iloc[0]["label"]))
print(results.iloc[0]["prediction"])

## Check Which Entities Were Misrecognized

In [ ]:
from collections import Counter
from tqdm.auto import tqdm

entity_counter = Counter()
examples = {}

for _, row in tqdm(list(results.iterrows()), total=len(results)):
    hyp_words = set(normalize(row["prediction"]).split())
    for entity in extract_entities(row["label"]):
        # the entity counts as recognized only if every of its words appears in the prediction
        if all(w in hyp_words for w in entity.split()):
            continue
        entity_counter[entity] += 1
        examples.setdefault(entity, {"label": row["label"], "prediction": row["prediction"],
                                     "source": row["source"]})

problem_words = pd.DataFrame(
    [(e, c, examples[e]["label"], examples[e]["prediction"], examples[e]["source"])
     for e, c in entity_counter.most_common()],
    columns=["word", "error_count", "example_label", "example_prediction", "example_source"],
)
problem_words.head(20)

In [ ]:
from collections import Counter

word_counter = Counter()
examples = {}

for _, row in results.iterrows():
    for etype, r, h in word_errors(row["label"], row["prediction"]):
        for w in r.split():
            if len(w) < 4:   # skip very short words
                continue
            word_counter[w] += 1
            examples.setdefault(w, {"label": row["label"], "prediction": row["prediction"],
                                    "source": row["source"]})

problem_words = pd.DataFrame(
    [(w, c, examples[w]["label"], examples[w]["prediction"], examples[w]["source"])
     for w, c in word_counter.most_common()],
    columns=["word", "error_count", "example_label", "example_prediction", "example_source"],
)
problem_words.head(20)

## Save the Ranked Word List

In [ ]:
problem_words.to_csv("tts_exercise/problem_words.csv", sep="\t", index=False)
print(f"Saved {len(problem_words)} problem words to tts_exercise/problem_words.csv")